In [ ]:
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


# Set dataset root depending on environment
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = "/content/drive/MyDrive/NN-kNN"
    DATA_ROOT = "/content/datasets"   # datasets inside Colab
    CHECKPOINTS = os.path.join(PROJECT_ROOT, "checkpoints")
    os.makedirs(DATA_ROOT, exist_ok=True)
    sys.path.append(PROJECT_ROOT)
    print("Running on Colab. DATA_ROOT =", DATA_ROOT)
else:
    PROJECT_ROOT = os.getcwd()
    DATA_ROOT = os.path.join(PROJECT_ROOT, "datasets")  # local ./datasets folder
    CHECKPOINTS = os.path.join(PROJECT_ROOT, "checkpoints")
    os.makedirs(DATA_ROOT, exist_ok=True)
    sys.path.append(PROJECT_ROOT)
    print("Running locally. DATA_ROOT =", DATA_ROOT)

Running locally. DATA_ROOT = g:\My Drive\NN-kNN\datasets


In [ ]:
# =====================
# Cell 1: Imports
# =====================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

from model.nnknn_model import NN_KNN_Model, train_model, default_args, GlocalFeatureWeight
import os
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
DATA_ROOT = os.environ.get("DATA_ROOT", "./datasets/")
os.makedirs(DATA_ROOT, exist_ok=True)

Using device: cuda


In [ ]:
# =====================
# Cell 2: Load MNIST
# =====================
# transform = transforms.Compose([
#     transforms.ToTensor(),  # (0,1) range
#     transforms.Normalize((0.1307,), (0.3081,))  # standard MNIST normalization
# ])

# mnist_train = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
# mnist_test  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# # Split training into train/val
# train_size = int(0.9 * len(mnist_train))
# val_size   = len(mnist_train) - train_size
# train_dataset, val_dataset = random_split(mnist_train, [train_size, val_size])

# batch_size = 64
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
# test_loader  = DataLoader(mnist_test, batch_size=batch_size, shuffle=False)

def compute_mean_std(dataset):
    loader = torch.utils.data.DataLoader(dataset, batch_size=1024, num_workers=2)
    mean, std, total_samples = 0.0, 0.0, 0
    for images, _ in loader:
        batch_samples = images.size(0)
        images = images.view(batch_samples, -1)
        mean += images.mean(1).sum(0)
        std += images.std(1).sum(0)
        total_samples += batch_samples
    mean /= total_samples
    std /= total_samples
    return mean.item(), std.item()

def MNIST(root=DATA_ROOT):
    initial_transform = transforms.Compose([transforms.ToTensor()])
    train_dataset = datasets.MNIST(root=root, train=True, download=True, transform=initial_transform)
    test_dataset = datasets.MNIST(root=root, train=False, download=True, transform=initial_transform)

    mean, std = compute_mean_std(train_dataset)
    print(f"MNIST mean={mean:.4f}, std={std:.4f}")

    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((mean,), (std,))])
    train_dataset = datasets.MNIST(root=root, train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST(root=root, train=False, download=True, transform=transform)

    X_train = torch.stack([train_dataset[i][0] for i in range(len(train_dataset))])
    y_train = torch.tensor([train_dataset[i][1] for i in range(len(train_dataset))])
    X_test = torch.stack([test_dataset[i][0] for i in range(len(test_dataset))])
    y_test = torch.tensor([test_dataset[i][1] for i in range(len(test_dataset))])
    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = MNIST()

MNIST mean=0.1307, std=0.3015


In [ ]:
X_train.shape

torch.Size([60000, 1, 28, 28])

In [ ]:
y_train

tensor([5, 0, 4,  ..., 5, 6, 8])

In [ ]:
debug_print = False
def dprint(*args):
  global debug_print
  if debug_print:
    print(*args)

In [ ]:
from model import feature_extractors
# conv_model = feature_extractors.CIFAR10Classifier()
conv_model = feature_extractors.MNISTClassifier()
feature_extractor = feature_extractors.get_feature_extractor_from(conv_model)
#move feature_extractor to device
feature_extractor.to(device)

Sequential(
  (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=3136, out_features=128, bias=True)
)

In [ ]:
# =====================
# Cell 5: Config
# =====================
cfg = default_args.copy()

dataset_name = "mnist"
cfg.update({
    "batch_size": 64,
    "sampling_cases_flag": True,
    "top_k_for_default_case_activation":20,
    "num_samples": 500,
    "glocal_fw_set_num": 1,
    "checkpoint_path": os.path.join(CHECKPOINTS, f'classifier_{dataset_name}.h5')
})


In [ ]:
cfg

{'task_type': 'classification',
 'softmax_over_cases': False,
 'tau': 1.0,
 'feature_dim': 128,
 'glocal_fw_set_num': 1,
 'training_epochs': 1000,
 'neg_weight_flag': False,
 'sampling_cases_flag': True,
 'use_sampling_cases_divisor': False,
 'sampling_cases_divisor': 100,
 'num_samples': 500,
 'case_activation_by_top_k_average': True,
 'top_k_for_default_case_activation': 20,
 'case_activation_default_percentage': 0.1,
 'bias_manual_set': False,
 'bias_manual_value': 6.0,
 'model_path': 'best_model.pth',
 'ignore_identical_in_training': True,
 'feature_extractor_lr': 0.0001,
 'glocal_weightor_lr': 0.001,
 'case_net_lr': 0.0001,
 'checkpoint_path': 'g:\\My Drive\\NN-kNN\\checkpoints\\classifier_mnist.h5',
 'batch_size': 64}

In [ ]:
# =====================
# Cell 6: Train Model
# =====================
best_acc, glocal_weightor, model = train_model(
    X_train, y_train,
    X_test, y_test,
    feature_extractor,
    cfg
)

print("Best validation accuracy:", best_acc)


Number of feature extractor parameters: 6
torch.Size([32, 1, 3, 3])
torch.Size([32])
torch.Size([64, 32, 3, 3])
torch.Size([64])
torch.Size([128, 3136])
torch.Size([128])
Number of glocal weightor parameters: 1
torch.Size([1, 128])
Number of case_net_params: 4
torch.Size([60000])
torch.Size([60000])
torch.Size([60000])
torch.Size([60000, 1])
*****************
Training started for training_epochs epochs with batch size 64
[Epoch 1/1000] - Loss: 0.0002
Epoch 1 - Validation Accuracy: 0.9812
Epoch 1 - Validation Loss: 0.0596
New best loss 0.0596 - Model saved.
[Epoch 2/1000] - Loss: 0.0049
Epoch 2 - Validation Accuracy: 0.9862
Epoch 2 - Validation Loss: 0.0450
New best loss 0.0450 - Model saved.
[Epoch 3/1000] - Loss: 0.0816
Epoch 3 - Validation Accuracy: 0.9882
Epoch 3 - Validation Loss: 0.0433
New best loss 0.0433 - Model saved.
[Epoch 4/1000] - Loss: 0.0987
Epoch 4 - Validation Accuracy: 0.9873
Epoch 4 - Validation Loss: 0.0425
New best loss 0.0425 - Model saved.
[Epoch 5/1000] - Loss: 

In [ ]:
# =====================
# Cell 8: Example Explanations
# =====================
test_loader = DataLoader(
    torch.utils.data.TensorDataset(X_test, y_test),
    batch_size=1, shuffle=True
)
model.explanation_mode = True
data, target = next(iter(test_loader))
data, target = data.to(device), target.to(device)
preds, classes, cases, labels, activs = model(data[:1])

print("True label:", target[0].item())
print("Predicted:", classes[0].item())
print("Top activated cases shape:", None if cases is None else cases[0].shape)


True label: 7
Predicted: 7
Top activated cases shape: torch.Size([4, 1, 28, 28])
